In [1]:
# !pip uninstall ultralytics torch torchvision torchaudio -y
!pip install torch torchvision torchaudio
!pip install ultralytics opencv-python matplotlib

Defaulting to user installation because normal site-packages is not writeable
  Using cached torchaudio-2.11.0-cp313-cp313-win_amd64.whl.metadata (6.9 kB)
Using cached torchaudio-2.11.0-cp313-cp313-win_amd64.whl (328 kB)



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
from ultralytics import YOLO
import cv2
import math


# Cargar modelo pose
model = YOLO("yolov8n-pose.pt")  # Modelo preentrenado de pose


# 2 Abrir webcam
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("No se pudo abrir la webcam")
    exit()

#Función para calcular distancia

def distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)


# Bucle principal

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Ejecutar inferencia
    results = model(frame, verbose=False)

    # Tomar primer resultado
    result = results[0]

    # Dibujar pose sobre la imagen
    annotated_frame = result.plot()

    # Verificar si hay keypoints detectados
    if result.keypoints is not None and len(result.keypoints.xy) > 0:

        # Solo trabajaremos con la primera persona detectada
        person = result.keypoints.xy[0]     # Coordenadas (17,2)
        confs = result.keypoints.conf[0]    # Confianzas (17,)

        # Índices COCO:
        # 0 Nose
        # 9 Left wrist
        # 10 Right wrist
        # 5 Left shoulder
        # 6 Right shoulder

        nose = person[0]
        left_wrist = person[9]
        right_wrist = person[10]
        left_shoulder = person[5]
        right_shoulder = person[6]

        nose_conf = confs[0]
        lw_conf = confs[9]
        rw_conf = confs[10]
        ls_conf = confs[5]
        rs_conf = confs[6]

        # Solo si los keypoints importantes tienen suficiente confianza
        if nose_conf > 0.4 and lw_conf > 0.4 and rw_conf > 0.4 and ls_conf > 0.4 and rs_conf > 0.4:

            # Convertir a enteros
            nose = (int(nose[0]), int(nose[1]))
            left_wrist = (int(left_wrist[0]), int(left_wrist[1]))
            right_wrist = (int(right_wrist[0]), int(right_wrist[1]))
            left_shoulder = (int(left_shoulder[0]), int(left_shoulder[1]))
            right_shoulder = (int(right_shoulder[0]), int(right_shoulder[1]))

            # Distancia entre hombros como referencia del tamaño del cuerpo
            shoulder_width = distance(left_shoulder, right_shoulder)

            # Distancias muñecas -> nariz
            dist_left = distance(left_wrist, nose)
            dist_right = distance(right_wrist, nose)

            # Dibujar puntos importantes
            cv2.circle(annotated_frame, nose, 6, (0, 255, 255), -1)
            cv2.circle(annotated_frame, left_wrist, 6, (255, 0, 0), -1)
            cv2.circle(annotated_frame, right_wrist, 6, (0, 0, 255), -1)

            # Regla: ambas manos cerca de la cabeza
            threshold = shoulder_width * 0.9

            if dist_left < threshold and dist_right < threshold:
                cv2.putText(
                    annotated_frame,
                    "Hands on Head Detected",
                    (30, 50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0, 255, 0),
                    3
                )
            else:
                cv2.putText(
                    annotated_frame,
                    "Pose Not Detected",
                    (30, 50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0, 0, 255),
                    3
                )

    # Mostrar video
    cv2.imshow("YOLO Pose - Hands on Head Detection", annotated_frame)

    # Salir con tecla q
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

The selected pose involves detecting when a person places both hands on their head.

This posture is relevant because it can represent a bodily action in different contexts, such as worry or stress.
Automatically detecting it is useful because it allows a system to interpret human behavior based on body posture, without the need for physical contact or additional devices.